# Assignment 9: Neural Networks

Fill in every `# TODO` and answer every question in the **Your answer** cells. Run all cells before submitting.

## Setup

We use the breast cancer data again (`diagnosis`: M = 1, B = 0), with the same 70/30 split as Assignment 8, so you can compare the neural networks with your logistic regression models.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

bc = pd.read_csv("https://bryantstats.github.io/math460/data/breast_cancer.csv")
bc = bc.drop(columns=["id", "Unnamed: 32"], errors="ignore")
y = (bc["diagnosis"] == "M").astype(int)
X = bc.drop(columns=["diagnosis"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)

def evaluate(name, model):
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name:<28} train accuracy = {train_acc:.3f}   test accuracy = {test_acc:.3f}")
    return test_acc

### Question 1. A baseline

Here is the L1 logistic regression from Assignment 8 as a baseline to beat.

In [ ]:
baseline = make_pipeline(StandardScaler(),
                         LogisticRegressionCV(penalty="l1", solver="liblinear", Cs=20, cv=5))
baseline.fit(X_train, y_train)
results = {"L1 logistic regression": evaluate("L1 logistic regression", baseline)}

### Question 2. Networks of different shapes

In scikit-learn, `hidden_layer_sizes=(2, 4)` means two hidden layers, with 2 and 4 neurons.

Train the following networks. For each one, use a pipeline with `StandardScaler()`, `MLPClassifier(..., max_iter=2000, random_state=1)`, and add its test accuracy to `results`.

a. Two hidden layers with 2 and 4 neurons.  
*(This and b. are the designs from the SAS version of this assignment.)*

b. Five hidden layers with 2, 4, 3, 2 and 1 neurons.

c. One hidden layer with 20 neurons.

In [ ]:
shapes = {"(2, 4)": (2, 4), "(2, 4, 3, 2, 1)": (2, 4, 3, 2, 1), "(20,)": (20,)}
nets = {}
for name, shape in shapes.items():
    # TODO: build and train the pipeline for this shape, store it in nets[name]
    results[name] = evaluate(name, nets[name])

### Question 3. Counting parameters

a. By hand, count the number of weights and biases in the `(2, 4)` network. The data has 30 predictors and the output layer has 1 neuron. Show your work.

b. Check your answer with the code below.

In [ ]:
def count_parameters(pipeline):
    mlp = pipeline[-1]
    return sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)

for name, net in nets.items():
    print(f"{name:<18} {count_parameters(net):>5} parameters")

**Your answer:** *Write your answer here.*

### Question 4. Is one run enough?

A neural network starts from **random weights**, and `random_state` sets that starting point. Retrain each network from Question 2 with `random_state = 0, 1, ..., 9` and record the 10 test accuracies.

a. Complete the function below, then run it for the three shapes.

b. What happens to the five-layer network in some runs? What does a test accuracy of 0.626 mean? (*Hint:* compute the proportion of benign tumors in `y_test`.)

c. Why is the `(2, 4, 3, 2, 1)` network so much less reliable than `(20,)`? Look at the size of its last hidden layers.

In [ ]:
def accuracy_over_seeds(shape, scale=True, seeds=range(10)):
    accs = []
    for seed in seeds:
        # TODO: train MLPClassifier(shape, max_iter=2000, random_state=seed), with a StandardScaler only if scale is True; append its test accuracy to accs
    return np.array(accs)

for name, shape in shapes.items():
    accs = accuracy_over_seeds(shape)
    print(f"{name:<18} mean = {accs.mean():.3f}  min = {accs.min():.3f}  max = {accs.max():.3f}   {np.round(accs, 3)}")

print("Proportion of benign tumors in the test set:", round((y_test == 0).mean(), 3))

**Your answer:** *Write your answer here.*

### Question 5. What scaling does for a neural network

Use `accuracy_over_seeds` with `scale=False` to repeat Question 4 **without** `StandardScaler` for the `(2, 4)` and `(20,)` networks. Compare with the scaled results and explain the difference.

In [ ]:
for name in ["(2, 4)", "(20,)"]:
    # TODO: compute the accuracies without scaling and print the mean and min
    print(f"{name:<8} without scaling: mean = {accs.mean():.3f}  min = {accs.min():.3f}   {np.round(accs, 3)}")

**Your answer:** *Write your answer here.*

### Question 6. The training curve

`MLPClassifier` stores the training loss after every iteration in `loss_curve_`. The code below trains the five-layer network with seed 0 and with seed 1. Plot both loss curves on one graph and describe how they differ.

In [ ]:
plt.figure(figsize=(8, 4.5))
for seed in [0, 1]:
    net = make_pipeline(StandardScaler(),
                        MLPClassifier((2, 4, 3, 2, 1), max_iter=2000, random_state=seed)).fit(X_train, y_train)
    acc = accuracy_score(y_test, net.predict(X_test))
    # TODO: plot net[-1].loss_curve_, labeled with the seed and the test accuracy
plt.xlabel("iteration")
plt.ylabel("training loss (cross-entropy)")
plt.yscale("log")
plt.title("Five-layer network (2, 4, 3, 2, 1)")
plt.legend()
plt.show()

**Your answer:** *Write your answer here.*

### Question 7. Your own design

Design your own network: choose the hidden layers, and optionally `alpha` (an L2 penalty, like Ridge), `activation` or `learning_rate_init`. Try at least **three** designs, report them in a table, and pick your best. Can you beat the L1 logistic regression on test accuracy?

*Fair comparison:* choose your design by cross-validation on the training data (for example with `cross_val_score`), not by looking at the test accuracy.

In [ ]:
from sklearn.model_selection import cross_val_score

designs = {
    # TODO: add at least three designs as name -> MLPClassifier(...)
}
cv_scores = {}
for name, mlp in designs.items():
    cv_scores[name] = cross_val_score(make_pipeline(StandardScaler(), mlp), X_train, y_train, cv=5).mean()
    print(f"{name:<24} 5-fold CV accuracy = {cv_scores[name]:.3f}")

In [ ]:
# TODO: refit your best design on all the training data and evaluate it on the test set
results["My design: " + best_name] = evaluate(best_name, best)
pd.Series(results, name="test accuracy").sort_values(ascending=False)

**Your answer:** *Write your answer here.*